Script to cross-reference protein-only genes against asthma/COPD/lung-function genetics (GWAS Catalog)

In [2]:
import pandas as pd

In [ ]:
# ---- folder that holds all the input files (edit this one line) -------------
path = "gwas_correlation/"
files = ["gwas_copd.tsv", "gwas_t2_high_asthma.tsv", "gwas_fev.tsv",
         "ver_10_jackson_goblet_concordance_full.xlsx",
         "ver_10_jackson_club_concordance_full.xlsx",
         "ver_10_jackson_ciliated_concordance_full.xlsx"]

goblet = pd.read_excel(path + files[3], sheet_name="Protein_only")
club   = pd.read_excel(path + files[4],   sheet_name="Protein_only")
cilia  = pd.read_excel(path + files[5], sheet_name="Protein_only")

goblet_genes = goblet["gene"].dropna().str.strip().str.upper()
club_genes   = club["gene"].dropna().str.strip().str.upper()
cilia_genes  = cilia["gene"].dropna().str.strip().str.upper()

def gwas_genes(filename):
    df = pd.read_csv(filename, sep="\t")
    genes = set()
    for x in ["MAPPED_GENE", "REPORTED GENE(S)"]:
        if x in df.columns:
            for value in df[x].dropna():
                text = str(value).replace(" x ", ",").replace(" - ", ",").replace(";", ",")
                for gene in text.split(","):
                    g = gene.strip().upper()
                    if g and g not in ("NR", "INTERGENIC") and not g.startswith("LOC"):
                        genes.add(g)
    return genes

gwas_asthma = gwas_genes(path + files[1]) # asthma
gwas_copd   = gwas_genes(path + files[0]) # COPD
gwas_fev    = gwas_genes(path + files[2]) # FEV/lung function

In [12]:
print("GWAS Catalog gene counts -> asthma:", len(gwas_asthma),
      "COPD:", len(gwas_copd), "FEV/lung function:", len(gwas_fev))

GWAS Catalog gene counts -> asthma: 26 COPD: 7 FEV/lung function: 2434


In [13]:
goblet_asthma = set(goblet_genes) & gwas_asthma

print(len(goblet_asthma))
print(goblet_asthma)

12
{'SLC22A5', 'RAD50', 'RANBP6', 'NCAPH', 'RPN1', 'HLA-DPB1', 'IL18R1', 'ATXN2', 'HLA-DPA1', 'D2HGDH', 'ITGB8', 'RNF39'}


In [14]:
# put the protein-only gene lists and GWAS gene lists into dictionaries
protein_only_genes = {
    "Goblet": set(goblet_genes),
    "Club": set(club_genes),
    "Multiciliated": set(cilia_genes)
}

gwas_sets = {
    "Asthma": gwas_asthma,
    "COPD": gwas_copd,
    "FEV / lung function": gwas_fev
}


# function to calculate N and %
def gwas_percent(protein_genes, gwas_genes):
    overlap = protein_genes & gwas_genes
    n = len(overlap)
    percent = (n / len(protein_genes)) * 100

    return f"{n} ({percent:.1f}%)"


# genes associated with any of the three GWAS categories
any_gwas = gwas_asthma | gwas_copd | gwas_fev

# make table
results = {}

for cell_type, protein_genes in protein_only_genes.items():
    results[cell_type] = {}
    for disease, gwas_genes in gwas_sets.items():
        results[cell_type][disease] = gwas_percent(
            protein_genes,
            gwas_genes
        )

    results[cell_type]["Total (any GWAS)"] = gwas_percent(
        protein_genes,
        any_gwas
    )

# convert into dataframe
gwas_summary = pd.DataFrame(results)

gwas_summary

,Goblet,Club,Multiciliated
Asthma,12 (0.3%),9 (0.2%),4 (0.2%)
COPD,0 (0.0%),0 (0.0%),0 (0.0%)
FEV / lung function,407 (8.6%),371 (8.4%),189 (8.4%)
Total (any GWAS),413 (8.8%),375 (8.5%),191 (8.5%)


In [15]:
overlap_details = []

for cell_type, protein_genes in protein_only_genes.items():
    for trait, gwas_genes in gwas_sets.items():
        overlap = sorted(protein_genes & gwas_genes)
        for gene in overlap:
            overlap_details.append({
                "Gene": gene,
                "Cell type": cell_type,
                "GWAS trait": trait
            })

gwas_gene_details = pd.DataFrame(overlap_details)

gwas_gene_details

,Gene,Cell type,GWAS trait
0,ATXN2,Goblet,Asthma
1,D2HGDH,Goblet,Asthma
2,HLA-DPA1,Goblet,Asthma
3,HLA-DPB1,Goblet,Asthma
4,IL18R1,Goblet,Asthma
...,...,...,...
987,VRK1,Multiciliated,FEV / lung function
988,XRCC5,Multiciliated,FEV / lung function
989,YWHAH,Multiciliated,FEV / lung function
990,ZNF391,Multiciliated,FEV / lung function
